# Notebook 02: Agro-Climatic Feature Engineering & Statistical Testing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fahadrahman444/scientific-research-and-methodologies/blob/main/colab_notebooks/02_Feature_Engineering_and_Statistical_Analysis.ipynb)

**Research Topic:** A Data-Driven Analysis of 15-Year Weather Variability and Its Impact on Agricultural Crop Yield in Bangladesh  
**Course:** CSE 418: Scientific Research and Methodologies Lab (UITS)  

---
## 1. Overview
This notebook computes agro-climatic indices, evaluates statistical hypotheses (Mann-Kendall test, Sen's slope, ADF stationarity), and prepares feature representations for predictive modeling.


In [ ]:
# Step 1: Environment & Data Ingestion
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

INTEGRATED_LOCAL = '../data/processed/bangladesh_faostat_weather_integrated_15yr.csv'
INTEGRATED_URL = f'{BASE_RAW_GITHUB}/data/processed/bangladesh_faostat_weather_integrated_15yr.csv'

df = pd.read_csv(INTEGRATED_LOCAL) if os.path.exists(INTEGRATED_LOCAL) else pd.read_csv(INTEGRATED_URL)
print(f'Loaded 15-year integrated dataset ({len(df)} records).')
df.head()


In [ ]:
# Step 2: Mann-Kendall Trend Test and Sen's Slope Implementation
def mann_kendall_test(x, alpha=0.05):
    n = len(x)
    s = 0
    for k in range(n - 1):
        for j in range(k + 1, n):
            s += np.sign(x[j] - x[k])
            
    var_s = (n * (n - 1) * (2 * n + 5)) / 18.0
    z = (s - 1) / np.sqrt(var_s) if s > 0 else ((s + 1) / np.sqrt(var_s) if s < 0 else 0)
    p_val = 2 * (1 - stats.norm.cdf(abs(z)))
    h = abs(z) > stats.norm.ppf(1 - alpha / 2)
    slopes = [(x[j] - x[k]) / (j - k) for k in range(n - 1) for j in range(k + 1, n)]
    return z, p_val, np.median(slopes), 'Statistically Significant' if h else 'No Monotonic Trend'

variables = ['yield_kg_ha', 'tmax', 'tmin', 'temp_mean', 'rain', 'heat_days']
mk_records = []
for var in variables:
    z, p, slope, status = mann_kendall_test(df[var].values)
    mk_records.append({
        'Variable': var,
        'Z_Score': round(z, 3),
        'p_value': round(p, 4),
        'Sens_Slope (unit/yr)': round(slope, 4),
        'Trend_Status': status
    })

mk_df = pd.DataFrame(mk_records)
print('=== MANN-KENDALL TREND TEST (15-YEAR BANGLADESH FAOSTAT SERIES) ===')
display(mk_df)


In [ ]:
# Step 3: De-trending Technological Gains (Separating Agronomy from Climate Shock)
# Fit linear / polynomial secular trend to capture HYV and irrigation adoption
x_years = df['year'].values - 2009
poly_fit = np.polyfit(x_years, df['yield_kg_ha'].values, deg=1)
df['yield_secular_trend'] = np.polyval(poly_fit, x_years)
df['detrended_climate_yield_residual'] = df['yield_kg_ha'] - df['yield_secular_trend']

plt.figure(figsize=(12, 5))
plt.plot(df['year'], df['yield_kg_ha'], 'o-', label='Observed FAOSTAT Yield (kg/ha)', color='#2ca02c')
plt.plot(df['year'], df['yield_secular_trend'], '--', label='Secular Trendline (Technological Adoption)', color='blue')
plt.title('De-trending Observed FAOSTAT Yield to Isolate Climate Signal', fontsize=13, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Yield (kg/ha)')
plt.legend()
plt.tight_layout()
plt.show()
